# Kinematische Analyse des Kuka KR6 R900-2 agilius

<figure>
<center>
<img width=500 src='figures/kr6.png' />
<figcaption>Kuka KR 6 R900-2</figcaption></center>
</figure>

Die Denavit-Hartenberg Parameter ergeben sich wie folgt:
<!--
\begin{array}{c|c|c|c|c}
  i & \theta_i & d_i & a_i & \alpha_i \\
  \hline
			1 & \theta_1 & 0,08920 & 0 & +90^\circ\\
			2 & \theta_2 & 0 & 0,425 & 0\\
			3 & \theta_3 & 0 & 0,392 & 0\\
			4 & \theta_4 & 0,10930 & 0 & -90^\circ\\
			5 & \theta_5 & 0,09475 & 0 & +90^\circ\\
			6 & \theta_6 & 0,08250 & 0 & 0
\end{array}
-->

\begin{array}{c|c|c|c|c}
  i & \theta_i & d_i & a_i & \alpha_i \\
  \hline
			1 & \theta_1 & d_1 & a_1 & +90^\circ\\
			2 & \theta_2 & 0 & a_2 & 0\\
			3 & \theta_3 & 0 & a_3 & +90^\circ\\
			4 & \theta_4 & d_4 & 0 & -90^\circ\\
			5 & \theta_5 & 0 & 0 & +90^\circ\\
			6 & \theta_6 & d_6 & 0 & 0
\end{array}

mit $d_1 = 0.400, a_1 = 0.025, a_2 = 0.455, a_3 = 0.025, d_4 = 0.420, d_6 = 0.090$

Die DH-Matrizen ergeben sich wie folgt:

In [1]:
from sympy import *
from IPython.display import display, Latex
from display_latex import *
init_printing(use_latex='mathjax')

In [2]:
def dhFrame(theta, d, a, alpha):
    
    rot_theta = Matrix([ [cos(theta), -sin(theta), 0, 0], 
                         [sin(theta), cos(theta),  0, 0], 
                         [0,             0,        1, 0], 
                         [0,             0,        0, 1] ])
    
    trans_d = Matrix([ [1, 0, 0, 0],
                       [0, 1, 0, 0],
                       [0, 0, 1, d],
                       [0, 0, 0, 1] ])
    
    trans_a = Matrix([ [1, 0, 0, a], 
                       [0, 1, 0, 0], 
                       [0, 0, 1, 0], 
                       [0, 0, 0, 1] ])
    
    rot_alpha = Matrix([ [1,          0,           0, 0], 
                         [0, cos(alpha), -sin(alpha), 0], 
                         [0, sin(alpha),  cos(alpha), 0], 
                         [0,          0,           0, 1] ])
    
    dh_frame = rot_theta * trans_d * trans_a * rot_alpha
    
    return dh_frame;

Wir lassen uns zuerst die allgemeine DH-Matrix $^{i-1}\mathbf{T}_i$ ausgeben:

In [3]:
theta_i, alpha_i, a_i, d_i = symbols('theta_i alpha_i a_i d_i')
Ti = symbols('{}^{i-1}\mathbf{T}_i')

Tdh = dhFrame(theta_i, d_i, a_i, alpha_i)

display_result(Ti, Tdh)

<IPython.core.display.Latex object>

Danach werden die Matrizen $^{0}\mathbf{T}_1$ bis $^{5}\mathbf{T}_6$ berechnet.

In [4]:
Ts = symbols('{}^0:7\mathbf{T}_0:7')

T = zeros(7,7)
for i in range(0, 7):
  for j in range(0, 7):
    if j+i > 6:
      break
    T[i,j+i] = Ts[8 * i + j]

T0 = ones(1,7)
T0[1] = T[0,1]
for i in range(2, 7):
  T0[i] = T0[i-1] * T[i-1,i]

display_result(T[0,6], T0[6])

<IPython.core.display.Latex object>

In [5]:
theta = symbols('theta_1:7')
d =  symbols('d_1:7')
a =  symbols('a_1:7')
T01 = dhFrame(theta[0], d[0], a[0], pi/2)

display_result(T[0,1], T01)

<IPython.core.display.Latex object>

In [6]:
T12 = dhFrame(theta[1], 0, a[1], 0)

display_result(T[1,2], T12)

<IPython.core.display.Latex object>

In [7]:
T23 = dhFrame(theta[2], 0, a[2], +pi/2)

display_result(T[2,3], T23)

<IPython.core.display.Latex object>

In [8]:
T34 = dhFrame(theta[3], d[3], 0, -pi/2)

display_result(T[3,4], T34)

<IPython.core.display.Latex object>

In [9]:
T45 = dhFrame(theta[4], 0, 0, pi/2)

display_result(T[4,5], T45)

<IPython.core.display.Latex object>

In [10]:
T56 = dhFrame(theta[5], d[5], 0, 0)

display_result(T[5,6], T56)

<IPython.core.display.Latex object>

Die gesamte Transformation von $K_0$ bis $K_6$ ergibt sich aus der Multipikaltion

In [12]:
T06 = T01 * T12 * T23 * T34 * T45 * T56

#display_result(T[0,6], T0[6], T06)
display_result(T06)

<IPython.core.display.Latex object>

was noch zusammengefasst werden kann

In [14]:
T06 = simplify(T06)

#display_result(T[0,6], T06)
display_result(T06)

<IPython.core.display.Latex object>

Mithilfe dieser Matrix lässt sich nun die **Vorwärtstransformation** bestimmen 

In [15]:
# Definition der Zahlenwerte der DH Parameter
d1 = 0.400
a1 = 0.025
a2 = 0.455
a3 = 0.025
d4 = 0.420
d6 = 0.09

def fkine(q): # q = [theta_1, ... , theta_6]
  T = T06.subs({d[0]:d1, a[0]:a1, a[1]:a2, a[2]:a3, d[3]:d4, d[5]:d6})
  for i in range(0, 6):
    T = T.subs({theta[i]:q[i]})
  T = T.evalf()
  return T

qz = Matrix([pi/2, pi/2, -pi/2, 0, 0, -pi/2])
#qz = Matrix([0, 0, 0, 0, 0, 0])
display_latex_result('\mathbf{q}', qz)
display_latex_result('\mathbf{f}(\mathbf{q})', fkine(qz))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

# Rücktransformation mittels geometrischem Ansatz


Die **Rücktransformation** (inverses kinematisches Problem) kann entweder als analytische Lösung oder als nummerische Lösung entwickelt werden. Die **analytische** Lösung ist nur für eine bestimmte Klasse von Roboterarmen möglich. Bei Robotern mit Zentralhand kann die Aufgabenstellung in zwei Teile aufgeteit werden, die dann getrennt gelöst werden. Die Zahlenwerte der Matrix ${}^0\mathbf{T}_6$ sind bei der Rücktransformation bekannt. Zuerst wird aus der Matrix 

$\newcommand{\mbf}{\mathbf}$
$$
{}^0\mbf{T}_6 \,=\,
\left( \begin{array}{cccc}
\mbf{x}_6 & \mbf{y}_6 & \mbf{z}_6 & \mbf{p}_6 \\
0 & 0 & 0 & 1
\end{array} \right) \,=\,
\left( \begin{array}{cccc}
\mbf{n} & \mbf{s} & \mbf{a} & \mbf{p}_6 \\
0 & 0 & 0 & 1
\end{array} \right) 
$$

die Position der Handwurzel der Zentralhand bestimmt:

<figure>
<center>
<img width=500 src='figures/RobHand.png' />
<figcaption>Berechnung der Position der Handwurzel </figcaption></center>
</figure>

Aus der Darstellung der einzelnen Vektoren in der Roboterhand erkennt man, dass sich hieraus der Positionsvektor $\mbf{p}_4$ von
${}^0\mbf{T}_4$ über die Beziehung

$$
\mbf{p}_4 \,=\, \mbf{p}_6 \,-\, d_6 \mbf{a}
$$
berechnen lässt.


In [16]:
p6 = T06[0:3,3]
display_latex_result('\mathbf{p}_{6}', p6)
#
avec = T06[0:3,2]
p4 = p6 - d[5]*avec
#display_latex_result('d_6', d[5])
p4 = simplify(p4)
display_latex_result('\mathbf{p}_{4}', p4)


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Aus dem symbolischen Ergebnis für $\mbf{p}_4$ ist direkt ersichtlich, dass die Position der Handwurzel nur von den ersten drei Gelenkvariablen abhängt. Bei der Rücktransformation sind die Zahlenwerte von ${}^0\mbf{T}_6$ und damit von $\mbf{p}_6$, $\mbf{a}$ und $\mbf{p}_4$ bekannt. Die Geometrie des Roboters kann nun genutzt werden, um aus der Position $\mbf{p}_4$ die ersten drei Gelenkwinkel zu bestimmen.

<figure>
<center>
<img width=500 src='figures/G60_g.png' />
<figcaption>Geometrie der unteren 3 Gelenkvariablen </figcaption></center>
</figure>

Die Parameter in der Grafik entsprechen $l_1 = a_1, l_2 = a_2, l_3 = a_3, l_4 = d_4, w = \mbf{p}_4$. 

$
\mbf{p}_4 = \begin{pmatrix}x_4\\y_4\\z_4\\\end{pmatrix}
$

Der Parameter $d_1$ ist in der Grafik nicht enthalten.

### Bestimmung von $\theta_1$

Der Winkel $\theta_1$ lässt sich direkt aus der Projektion der Robotergeometrie ($\mbf{p}_4$) auf die $x_0$, $y_0$ Ebene gewinnen:

$
\theta_1 = \mathrm{atan2(y_4, x_4)}
$

Eine zweite Lösung ergibt sich durch die Drehung von $\theta_1$ um $180^°$ und „Kippen des Arms auf die andere Seite“. Dies kann durch eine Konfigurationsvariable

$
K_1 \,=\, \left\{ \begin{array}{ll}
+1 & \mbox{vorne} \\
-1 &  \mbox {180° gekippt}
\end{array} \right.
$

erreicht werden

$
\theta_1 = \mathrm{atan2(K_1 \cdot y_4, K_1 \cdot x_4)}
$

### Bestimmung von $\theta_2$

Betrachtet man den Positionsvektor $\mbf{p}_4$ nun in der $\mbf{x}_1,\mbf{y}_1$-Ebene, so erhält man die geometrischen
Zusammenhänge.

<figure>
<center>
<img width=500 src='figures/theta_2_bestimmung.png' />
<figcaption>Berechnung von $\theta_2$ </figcaption></center>
</figure>


Für die eingetragene Längen $l_2$, $l_3$ und für die Winkel $\alpha$ und $\beta$ lassen sich die folgenden Beziehungen herleiten:

$
{\displaystyle l_1 = z_4 - d_1}
$

$
{\displaystyle l_2 = \sqrt{x_4^2+y_4^2} -K_1 \cdot a_1}
$

$
{\displaystyle l_3 = \sqrt{l_2^2 + l_1^2}}
$

$
{\displaystyle l_4 = \sqrt{d_4^2+a_3^2}}
$

$
{\displaystyle \cos\alpha = \frac{l_2}{l_3}}
$

$
{\displaystyle \sin\alpha = \frac{l_1}{l_3}}
$

mithilfe des Kosinussatzen ${\displaystyle c^{2}=a^{2}+b^{2}-2\cdot a\cdot b\cdot \cos \gamma }$

$
{\displaystyle \cos\beta = \frac{a_2^2+l_3^2-l_4^2}{2a_2 l_3}}
$

$
{\displaystyle \sin\beta = \sqrt{1-\cos^2 \beta}}
$

Der gesuchte Gelenkwinkel $\theta_2$ etzt sich aus den beiden Winkeln $\alpha$ und $\beta$ zusammen:

Fall vorne, Oberer Arm: $\theta_2 = \alpha+\beta$

mit den Additionstheoreme

$
{\displaystyle \sin(\alpha \pm \beta )=\sin \alpha \cdot \cos \beta \pm \cos \alpha \cdot \sin \beta } ,
$

$
{\displaystyle \cos(\alpha \pm \beta )=\cos \alpha \cdot \cos \beta \mp \sin \alpha \cdot \sin \beta } ,
$

können die Gleichungen umgeformt werden

$
\begin{array}{lcl}
\sin\theta_2  & = &  \sin\alpha\cos\beta  + \cos\alpha\sin\beta \\
\cos\theta_2 & = & \cos\alpha\cos\beta - \sin\alpha\sin\beta
\end{array}
$

Fall vorne, Unterer Arm: $\theta_2 = \alpha-\beta$

$
\begin{array}{lcl}
\sin\theta_2  & = & \sin\alpha\cos\beta - \cos\alpha\sin\beta \\
\cos\theta_2 & = & \cos\alpha\cos\beta + \sin\alpha\sin\beta
\end{array}
$

Fall gekippt, Oberer Arm: $\theta_2 = -\alpha-\beta$

$
\begin{array}{lcl}
\sin\theta_2  & = & -\sin\alpha\cos\beta - \cos\alpha\sin\beta \\
\cos\theta_2 & = & \cos\alpha\cos\beta - \sin\alpha\sin\beta
\end{array}
$

Fall gekippt, Unterer Arm: $\theta_2 = -\alpha+\beta$

$
\begin{array}{lcl}
\sin\theta_2   & = & -\sin\alpha\cos\beta + \cos\alpha\sin\beta \\
\cos\theta_2  & = & \cos\alpha\cos\beta + \sin\alpha\sin\beta
\end{array}
$

Durch die Einführung des zweiten Konfigurationsindikators $K_2$ mit

$
K_2 \,=\, \left\{ \begin{array}{ll}
+1 & \mbox{Oberer Arm} \\
-1 &  \mbox {Unterer Arm}
\end{array} \right.
$

lassen sich dann zusammen mit $K_1$ diese obigen vier Fälle zusammenfassen:

$
\begin{array}{lcl}
\sin\theta_2 & = & K_1 \,\sin\alpha\cos\beta \,+\, K_2 \cos\alpha\sin\beta \\
\cos\theta_2 & = & \cos\alpha\cos\beta \,-\, K_2 \sin\alpha\sin\beta 
\end{array}
$

und somit schließlich

$
\theta_2 = \mathrm{atan2}(\sin \theta_2, \cos\theta_2)
$

### Bestimmung des Winkels $\theta_3$

Den Betrachtungen zu den Winkeln $\theta_1$ und $\theta_2$ entsprechend wird zur Bestimmung von $\theta_3$ der Positionsvektor
$\mbf{p_4}$ jetzt in der \$\mbf{x}_2,\mbf{y}_2$-Ebene betrachtet und man erhält die geometrischen Zusammenhänge


<figure>
<center>
<img width=300 src='figures/theta3_berechnung_1.png' />
<figcaption>Berechnung von $\theta_3$, Fall 1 </figcaption></center>
</figure>

Aus den Abbildungen erkennbar, 

Es gelten unabhängig von der Konfiguration die folgenden Gleichungen für die Winkel $\alpha$ und $\beta$

$
{\displaystyle \cos\alpha = \frac{a_2^2+l_4^2 - l_3^2}{2 a_2 l_4}}
$

$
{\displaystyle \sin\alpha = \sqrt{1-\cos^2\alpha}}
$

$
{\displaystyle \cos\beta = \frac{a_3}{l_4}}
$

$
{\displaystyle \sin\beta = \frac{d_4}{l_4}}
$

Der gesuchte Gelenkwinkel $\theta_3$ setzt sich aus den beiden Winkeln $\alpha$ und $\beta$ zusammen:

Fall 1: Vorne, Oberer Arm und hinten, Unterer Arm: $\theta_3 = \alpha + \beta - \pi$

Die Gleichungen $\sin(\theta_3) = \sin(\alpha + \beta - \pi)$ und $\cos(\theta_3) = \cos(\alpha + \beta - \pi)$ lassen sich vereinfachen und mittels Additionstheoremen aufspalten:

In [17]:
th, al, be = symbols('theta alpha beta')
# für den Fall 1 vorne oberer Arm, hinten unterer Arm
s_theta = sin(al + be -pi)
display_latex_result('\\sin(\\theta) = \\sin(\\alpha + \\beta - \\pi)', s_theta)
s_theta = -(sin(al)*cos(be) + cos(al)*sin(be))
display_result(simplify(s_theta), s_theta)

c_theta = cos(al + be -pi)
display_latex_result('\\cos(\\theta) = \\cos(\\alpha + \\beta  - \\pi)', c_theta)
c_theta = -(cos(al)*cos(be) - sin(al)*sin(be))
display_result(c_theta, simplify(c_theta))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<figure>
<center>
<img width=300 src='figures/theta3_berechnung_2.png' />
<figcaption>Berechnung von $\theta_3$, Fall 2 </figcaption></center>
</figure>

Fall2: vorne, Unterer Arm und hinten, Oberer Arm: $\theta_3 = \beta + \pi - \alpha$

In [18]:
s_theta = sin(be + pi -al)
display_latex_result('\\sin(\\theta) = \\sin(\\beta + \\pi - \\alpha)', s_theta)
s_theta = sin(al)*cos(be) - cos(al)*sin(be)
display_result(simplify(s_theta), s_theta)

c_theta = cos(be + pi -al)
display_latex_result('\\cos(\\theta) = \\cos(\\beta + \\pi - \\alpha)', c_theta)
c_theta = -(cos(al)*cos(be) + sin(al)*sin(be))
display_result(c_theta, simplify(c_theta))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Fälle vorne, Unterer Arm und hinten, Oberer Arm: $\theta_3 = \beta + \pi - \alpha$

Mit Hilfe der beiden eingeführten Konfigurationsindices ergibt sich dann

$
\begin{array}{lcl}
\sin\theta_3  & = & -K_1 K_2 \sin\alpha\cos\beta -\cos\alpha\sin\beta \\
\cos\theta_3 & = &  + K_1 K_2 \sin\alpha\sin\beta - \cos\alpha\cos\beta 
\end{array}
$

und somit

$
\theta_3 = \mathrm{atan2}(\sin \theta_3, \cos\theta_3)
$

### Bestimmung der Winkel $\theta_4 \dots \theta_6$

Nachdem die ersten drei Gelenkwinkel bestimmt worden sind, ist die
Transformationsmatrix ${}^0\mbf{T}_3$ vollständig und nummerisch bekannt:
<!--
$
{}^0\mbf{T}_3  = 
\left( \begin{array}{cccc}
\mbf{x}_3 & \mbf{y}_3 & \mbf{z}_3 & \mbf{p}_3 \\
0 & 0 & 0 & 1
\end{array} \right)
$
-->

In [19]:
T03 = T01 * T12 * T23
T03 = simplify(T03)
display_latex_result('{}^0\mathbf{T}_{3}', T03)

<IPython.core.display.Latex object>

Es kann aus ${}^0\mathbf{T}_{3}$ und aus der Zielmatrix ${}^0\mathbf{T}_{6}$ die Orientierung der oberen drei Winkel bestimmt werden:

$
{}^0\mathbf{T}_{3} \cdot {}^4\mathbf{T}_{6} = {}^0\mathbf{T}_{6} \; \Rightarrow \; {}^4\mathbf{T}_{6} = {}^0\mathbf{T}_{3}^{-1} \cdot {}^0\mathbf{T}_{6} 
$

die Matrix

$
%\mbf{T} \,=\, 
{}^0\mathbf{T}_{3}^{-1} \cdot {}^0\mathbf{T}_{6} \,=\, 
\left( \begin{array}{cccc}
n_x & o_x & a_x & p_x \\
n_y & o_x & a_y & p_y \\
n_z & o_x & a_z & p_z \\
0 & 0 & 0 & 1
\end{array} \right) 
$

kann nummerisch berechnet werden und die Lösung der oberen drei Gelenkwinkel kann durch Koeffizientenvergleich erfolgen, wobei nur der Rotationteil betrachtet werden muss.

In [20]:
T46 = simplify(T34 * T45 * T56)
display_latex_result('{}^4\mathbf{T}_{6}', T46)

<IPython.core.display.Latex object>

Es lassen sich nun die Beziehungen der Gelenkvariablen direkt durch elementweisen Koeffizientenvergleich ablesen

$
\cos\theta_5 = a_z
$

$
\sin\theta_5 = k_3 \sqrt{1 - \cos^2\theta_5}
$

Für den Fall $a_z = 1$ ($\sin\theta_5 = 0$) liegt eine singuläre Stellung vor, $\theta_4$ ist dann frei wählbar, sonst

$
\displaystyle{\cos\theta_4 = \frac{a_x}{\sin\theta_5 }}
$

$
\displaystyle{\sin\theta_4 = \frac{a_y}{\sin\theta_5 }}
$




$\theta_6$ lässt sich aus den Elementen $n_x, n_y, o_x, o_y$ bestimmen.

In [21]:
c_theta_6 = simplify(T46[1,1]*cos(theta[3]) - T46[0,1]*sin(theta[3]))
display_latex_result('\\cos(\\theta_4) o_y - \\sin(\\theta_4) o_x', c_theta_6)

s_theta_6 = simplify(T46[1,0]*cos(theta[3]) - T46[0,0]*sin(theta[3]))
display_latex_result('\\cos(\\theta_4) n_y - \\sin(\\theta_4) n_x', s_theta_6)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

### Implementierung Rücktransformation

In [22]:
# Implementierung der Rückwärtstransformation
# d_1 = 0.400, a_1 = 0.025, a_2 = 0.455, a_3 = 0.025, d_4 = 0.420, d_6 = 0.090
def ikine(T, K = Matrix([1, 1, 1])): # q = [theta_1, ... , theta_6]
  q = Matrix([0, 0, 0, 0, 0, 0])
  T = T.evalf()
  p6 = T[0:3,3]
  #display_latex_result('\mathbf{p}_{6}', p6)
  avec = T[0:3,2]
  p4 = p6 - d6*avec
  #display_latex_result('\mathbf{p}_{4}', p4)
  x4 = p4[0]
  y4 = p4[1]
  z4 = p4[2]
  
  # theta1
  q[0] = atan2(K[0] * y4, K[0] * x4)
  
  # theta2
  l1 = z4 - d1
  l2 = sqrt(x4*x4 + y4*y4) -K[0] * a1  
  l3 = sqrt(l1*l1 + l2*l2)
  l4 = sqrt(d4*d4 + a3*a3)
  c_alpha = l2 / l3
  s_alpha = l1 / l3
  c_beta = (a2*a2 + l3*l3 - l4*l4)/(2*a2*l3)
  s_beta = sqrt(1 - c_beta*c_beta)
  s_theta2 = K[0]*s_alpha*c_beta + K[1]*c_alpha*s_beta  
  c_theta2 = c_alpha*c_beta - K[1]*s_alpha*s_beta  
  q[1] = atan2(s_theta2, c_theta2)
  
  # theta3  
  c_alpha = (a2*a2 + l4*l4 - l3*l3)/(2*a2*l4)
  s_alpha = sqrt(1 - c_alpha*c_alpha)
  c_beta = a3 / l4
  s_beta = d4 / l4
  s_theta3 = -K[0]*K[1]*s_alpha*c_beta - c_alpha*s_beta  
  c_theta3 = K[0]*K[1]*s_alpha*s_beta - c_alpha*c_beta   
  #alpha = atan2(s_alpha, c_alpha)
  #beta = atan2(s_beta, c_beta)
  #q[2] = -pi +alpha +beta  
  q[2] = atan2(s_theta3, c_theta3)

  # theta4 .. theta6
  T03n = T03.subs({d[0]:d1, a[0]:a1, a[1]:a2, a[2]:a3, d[3]:d4, d[5]:d6})
  for i in range(0, 3):
    T03n = T03n.subs({theta[i]:q[i]})
  T03n = T03n.evalf()

  T46n = T03n.inv() * T
    
  # theta 5
  c_theta5 = T46n[2,2]
  s_theta5 = K[2]*sqrt(1 - c_theta5*c_theta5)  
  q[4] = atan2(s_theta5, c_theta5)
    
  # check singularity
  if abs(s_theta5) < 0.0000001:
    q[4] = 0
    q[3] = 0 # frei wählbar
    c_theta4 = 1
    s_theta4 = 0
  else:
    c_theta4 = T46n[0,2] / s_theta5
    s_theta4 = T46n[1,2] / s_theta5
    q[3] = atan2(s_theta4, c_theta4)
  
  c_theta6 = c_theta4 * T46n[1,1] - s_theta4 * T46n[0,1]  
  s_theta6 = c_theta4 * T46n[1,0] - s_theta4 * T46n[0,0]  

  q[5] = atan2(s_theta6, c_theta6)
        
  q = q.evalf()
  return q

In [23]:
# Bestimmung der Konfiguration aus der Gelenkstellung
import numpy as np

def getConf(q):
  T = T06.subs({d[0]:d1, a[0]:a1, a[1]:a2, a[2]:a3, d[3]:d4, d[5]:d6})
  for i in range(0, 6):
    T = T.subs({theta[i]:q[i]})
  T = T.evalf()
  p6 = T[0:3,3]
  #display_latex_result('\mathbf{p}_{6}', p6)
  avec = T[0:3,2]
  p4 = p6 - d6*avec
  K = Matrix([0, 0, 0])
  c_theta1 = cos(q[0])  
  s_theta1 = sin(q[0])  
  if c_theta1 != 0:
    K[0] = np.sign(p4[0] / c_theta1)
  else:
    K[0] = np.sign(p4[1] / s_theta1)

  l1 = p4[2] - d1
  l2 = sqrt(p4[0]*p4[0] + p4[1]*p4[1]) -K[0] * a1  
  l3 = sqrt(l1*l1 + l2*l2)
  l4 = sqrt(d4*d4 + a3*a3)
  c_alpha = l2 / l3
  s_alpha = l1 / l3
  c_beta = (a2*a2 + l3*l3 - l4*l4)/(2*a2*l3)
  s_beta = sqrt(1 - c_beta*c_beta)
  beta = atan2(s_beta,c_beta)
  display_latex_result('\\beta', beta)
  alpha = atan2(s_alpha, c_alpha)
  display_latex_result('\\alpha', alpha)
  display_latex_result('\\theta_2', q[1])
  #s_theta2 = K[0]*s_alpha*c_beta + K[1]*c_alpha*s_beta  
  c_theta2 = cos(q[1])
  K[1] = np.sign((c_alpha*c_beta - c_theta2) / (s_alpha*s_beta))

#  s_theta2 = K[0]*s_alpha*c_beta + K[1]*c_alpha*s_beta  
#  c_theta2 = c_alpha*c_beta - K[1]*s_alpha*s_beta  
    
  K[2] = np.sign(q[4])
  if K[2] == 0:
    K[2] = 1
    
  K = K.evalf()
  return K

# qz = Matrix([0.1, -pi/2 , 0, 0, 0, -pi/2])
# getConf(qz)


In [24]:
qz = Matrix([pi, pi/2, pi, -0.1, -0.1, 0])
#qz = Matrix([-0.1, 0.1, 0.1, 0.2, -0.8, 0.6])
display_latex_result('\mathbf{q}', qz)
Ttcp = fkine(qz)
display_latex_result('\mathbf{f}(\mathbf{q})', Ttcp)

K = getConf(qz) 
#K = Matrix([-1, 1, -1])
display_latex_result('\mathbf{K}', K)
qq = ikine(Ttcp, K)
display_latex_result('\mathbf{q}', qq)

#qz = Matrix([0, 0, 0, 0, 0, pi/2])
#display_latex_result('\mathbf{q}', qz)
#display_latex_result('\mathbf{f}(\mathbf{q})', fkine(qz))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

# Rücktransformation mittels nummerischen Ansatz

Für die Rücktransformation wird die geometrische Jacobi-Matrix bestimmt.

$$
\dot{\mathbf{x}} = \mathbf{J}_\mathrm{g}(\mathbf{q}) \cdot \dot{\mathbf{q}}
$$

mit $\dot{\mathbf{x}} = (v_x, v_y, v_z, \omega_x, \omega_y, \omega_z)^\mathrm{T}$ und $\dot{\mathbf{q}} = (\dot{q}_1, \dot{q}_2, \dot{q}_3, \dot{q}_4, \dot{q}_5, \dot{q}_6)^\mathrm{T}$


Das erste Gelenk ist ein Drehgelenk, die Drehung erfolgt um die $\mathbf{z}_0$-Achse, die erste Spalte der Jacobi-Matrix $\mathbf{J}_1$ ergibt sich deshalb wie folgt:
$$ 
\mathbf{J}_1^\mathrm{R} = 
\begin{pmatrix}
	\mathbf{J}_\mathrm{v}\\
	\mathbf{J}_\omega
\end{pmatrix}
= 
\begin{pmatrix}
	\mathbf{z}_0 \times (\mathbf{p}_{6} - \mathbf{p}_{0})\\
	\mathbf{z}_{0}
\end{pmatrix}
$$

mit

$$
{}^0\mathbf{T}_{0} = 
\begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0  \\
0 & 0 & 0 & 1
\end{pmatrix}
= 
\begin{pmatrix}
\mathbf{x}_{0} & \mathbf{y}_{0} & \mathbf{z}_{0} & \mathbf{p}_{0} \\
0 & 0 & 0 & 1
\end{pmatrix}
$$

und 

$$
{}^0\mathbf{T}_{6} = {}^0\mathbf{T}_{1} \cdot {}^1\mathbf{T}_{2} \cdot 
{}^2\mathbf{T}_{3} \cdot {}^3\mathbf{T}_{4} \cdot {}^4\mathbf{T}_{5} \cdot 
{}^5\mathbf{T}_{6} = 
\begin{pmatrix}
\mathbf{x}_{6} & \mathbf{y}_{6} & \mathbf{z}_{6} & \mathbf{p}_{6} \\
0 & 0 & 0 & 1
\end{pmatrix}
$$


In [25]:
J = symbols('\mathbf{J}_1:7')
Jgg = symbols('\mathbf{J}_g')
Jgs = Matrix(1,6,[J[0], J[1], J[2], J[3], J[4],J[5]])

display_result(Jgg, Jgs)

<IPython.core.display.Latex object>

In [27]:
T00 = eye(4)     # Transformation von K0 nach K0 ist I4x4
z0 = T00[:3,2]
p0 = T00[:3,3]
p6 = T06[:3,3]
Jg = zeros(6,6)
Jg[:,0] =  simplify(Matrix([z0.cross(p6 - p0), z0]))

display_result(J[0], Jg[:,0])

<IPython.core.display.Latex object>

Das Ergebnis von $\mathbf{J}_1$ zeigt, dass bei Bewegung des ersten Gelenks ($\theta_1$), sich die Position in der z-Komponente nicht ändert und dass die Orientierung sich nur um die z-Achse verändert.


In [28]:

#Die weiteren fünf Achsen werden analog dazu berechnet:

Tm = [T00, T01, T12, T23, T34, T45]
T0i = Tm[0]
for i in range(1, 6):
  T0i = T0i * Tm[i]
  T0i = simplify(T0i)
  z_i = T0i[:3,2]
  p_i = T0i[:3,3]
  Jg[:,i] =  simplify(Matrix([z_i.cross(p6 - p_i), z_i]))

  display_result(J[i], Jg[:,i])

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Bei Bewegung des letzten Gelenks wid die Position nicht mehr geändert.

Zum Schluss geben wir die nun vollständige Jacobi-Matrix $\mathbf{J}_g$ aus:

In [29]:
display_result(Jgg, Jgs)
display_result(Jg)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

---
Mithilfe der Jacobi-Matrix lässt sich nun die Rücktransformation numerisch bestimmen. Ausgehend von einem Startpunkt der Gelenkwinkel $\mathbf{q}_k$ kann die Lösung für $\mathbf{q} = \mathbf{f}^{-1}(\mathbf{x})$ iterativ mittels Gauß-Newton-Verfahren bestimmt werden:
$$
\mathbf{q}_{k+1} = \mathbf{q}_k + \mathbf{J}^{-1}(\mathbf{q}_k) \cdot (\mathbf{x} - \mathbf{x}_k) \quad \text{mit} \quad \mathbf{x}_k = \mathbf{f}(\mathbf{q}_k)
$$

Sobald eine hinreichend genaue Lösung ($\left| \mathbf{x} - \mathbf{x}_k \right| < \epsilon$) für $\mathbf{x}$ gefunden ist, wird die Iteration abgebrochen. 


In [30]:
def mod_2pi(a): # -pi < a < pi
  a = a % (2*pi)
  if a < -pi:
    a += 2*pi 
  if a > pi:
    a -= 2*pi
  return a 

def tr2delta(T0, T1): # Bewegungsdelta von T0 -> T1, siehe auch tr2delta aus Robotics Toolbox Peter Corke
  p = T1[:3, 3]    # Zielposition
  R = T1[:3, :3]   # Zielorientierung als Rotationsmatrix
  p0 = T0[:3,3]    # Ausgangs-Position in T0
  R0 = T0[:3,:3]   # Ausgangs-Orientierung in T0 
  dp = (p - p0)    # Positionsdelta
  dR = R * R0.transpose()
  do = 0.5 * Matrix([dR[2,1] - dR[1,2], dR[0,2] - dR[2,0], dR[1,0] - dR[0,1]]) # Orientierungsdelta
  # do = 0.5 * (R0[:,0].cross(R[:,0]) + R0[:,1].cross(R[:,1]) + R0[:,2].cross(R[:,2])) 
  dx = Matrix([dp, do]).evalf() # Delta in der Pose   
  return dx

def iikine(T, qk): # T homogene Transformationsmatrix als Ziel 
  Js = Jg.subs({d[0]:0.345, a[0]:0.020, a[1]:0.260, a[2]:0.020, d[3]:0.260, d[5]:0.075})
  for k in range(0, 30):
    Tk = fkine(qk)
    dx = tr2delta(Tk, T)
    norm_x = sqrt(dx.dot(dx))
    if norm_x < 0.000001:
      print('\n Abbruch bei k = ' + str(k) + '\n')
      for j in range(0, 6):  
        qk[j] = mod_2pi(qk[j]) # -pi < theta_i < pi
      break
    Jk = Js
    for i in range(0, 6):
      Jk = Jk.subs({theta[i]:qk[i]})
    Jkinv = (Jk.evalf()).inv()
    qk = qk + Jkinv * dx
    qk = qk.evalf()
    #print(qk)
  return qk.evalf()  

#
q = Matrix([-pi/2, 0, 0, pi/2, 0, -pi/2])
display_latex_result('\mathbf{q}', q)
qk = Matrix([-0.5, -0.5, -0.5, -0.5, -0.1, -0.5])
T = fkine(q)
display_latex_result('\mathbf{T} = \mathbf{f}(\mathbf{q})', T)
ql = ikine(T, qk)
display_latex_result('\mathbf{q} = \mathbf{f}^{-1}(\mathbf{T})', ql.evalf(6))
Tn = fkine(ql)
display_latex_result('\mathbf{T} = \mathbf{f}(\mathbf{q})',Tn.evalf(6))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>